# Offline Action Mode Probe For SmolVLA

Notebook này dùng để kiểm tra lỗi kiểu **cùng một observation/start state nhưng policy sinh nhiều mode hành vi khác nhau**: thành công, đi quá cốc, hoặc đi về gần safe/start pose.

Nó chạy hoàn toàn offline, không cần kết nối robot. Mặc định dùng frame từ dataset đã upload/download trên Hugging Face. Nếu có `recorded_obs` từ lần infer fail, có thể thay cell chọn observation bằng dữ liệu đó sau.

## 0. Parameters

- `N_SAMPLES`: số lần chạy cùng một observation với seed khác nhau.
- `PROBE_EPISODES`: các episode/frame dùng làm observation probe.
- `CLUSTER_STEPS`: số timestep đầu của action chunk dùng để cluster mode.

In [ ]:
from pathlib import Path
import gc
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CACHE_DIR = Path(os.environ.get("XAI_MODE_PROBE_CACHE", "xai_mode_probe_cache")).expanduser()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_REPO = "di-techinnova/smolvla-pouring-0.3-cutted"
NEW_MODEL_REV = "f7029d03d69e149cb4b7cea8747d7158d35a8fd0"
OLD_MODEL_REV = "b6f2aafdbdd793046747fad8207459402c33c4b0"

DATASET_REPO = "di-techinnova/so-arm-101-pouring-0.3-cutted"
DATASET_REV = "1e24a69dbf1fbf602bb71f47f47043d2c2756eaf"
VIDEO_BACKEND = "pyav"

TASK = "Pour from orange cup into blue cup."
SAFE_POSE = np.array([-5.626, -104.132, 96.615, 73.890, 0.308, 0.410], dtype=np.float64)
JOINT_NAMES = ["shoulder_pan.pos", "shoulder_lift.pos", "elbow_flex.pos", "wrist_flex.pos", "wrist_roll.pos", "gripper.pos"]

PROBE_EPISODES = [0, 1, 2, 10, 50, 51, 120, 145]
PROBE_FRAME_INDEX = 0
N_SAMPLES = 100
SEED_BASE = 1000
CLUSTER_STEPS = 12
N_CLUSTERS = 3

# Wider phase sweep. This is lighter per observation than the deep probe above,
# but covers many more start/approach frames in one full notebook run.
PHASE_SWEEP_EPISODES = PROBE_EPISODES
PHASE_SWEEP_FRAME_INDICES = [0, 5, 10, 15, 20, 25, 30, 45, 60]
PHASE_SWEEP_N_SAMPLES = 25
STAY_CURRENT_DIST_THRESH = 8.0
SAFE_LIKE_DIST_THRESH = 30.0
SAFE_PULL_MARGIN = 10.0
OVERSHOOT_FIRST_DIST_THRESH = 30.0

print("DEVICE", DEVICE)
print("CACHE_DIR", CACHE_DIR.resolve())

## 1. Download / Resolve Snapshots

In [ ]:
def dl_snapshot(repo_id: str, repo_type: str, revision: str, local_name: str) -> Path:
    local_dir = CACHE_DIR / local_name
    path = snapshot_download(
        repo_id=repo_id,
        repo_type=repo_type,
        revision=revision,
        local_dir=local_dir,
    )
    return Path(path)

new_model_dir = dl_snapshot(MODEL_REPO, "model", NEW_MODEL_REV, "new_model")
old_model_dir = dl_snapshot(MODEL_REPO, "model", OLD_MODEL_REV, "old_model")
dataset_dir = dl_snapshot(DATASET_REPO, "dataset", DATASET_REV, "dataset")

print("new_model_dir", new_model_dir)
print("old_model_dir", old_model_dir)
print("dataset_dir", dataset_dir)

## 2. Load Dataset And Select Probe Observations

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset(DATASET_REPO, root=dataset_dir, video_backend=VIDEO_BACKEND)
print(ds)

data = ds.hf_dataset.to_pandas()
probe_rows = []
for ep in PROBE_EPISODES:
    rows = data[(data["episode_index"] == ep) & (data["frame_index"] == PROBE_FRAME_INDEX)]
    if rows.empty:
        print("missing episode/frame", ep, PROBE_FRAME_INDEX)
        continue
    probe_rows.append(rows.iloc[0].to_dict())

probe_df = pd.DataFrame(probe_rows)
probe_df[["index", "episode_index", "frame_index", "timestamp"]]

In [ ]:
def show_probe(global_index: int):
    item = ds[int(global_index)]
    state = item["observation.state"].detach().cpu().numpy()
    print("state", np.round(state, 3))
    print("dist_state_safe", round(float(np.linalg.norm(state - SAFE_POSE)), 3))
    cams = [k for k in item if k.startswith("observation.images")]
    fig, axes = plt.subplots(1, len(cams), figsize=(5 * len(cams), 3))
    if len(cams) == 1:
        axes = [axes]
    for ax, cam in zip(axes, cams):
        img = item[cam].detach().cpu()
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = img.permute(1, 2, 0)
        ax.imshow(img.numpy())
        ax.set_title(cam)
        ax.axis("off")
    plt.show()
    return item

first_item = show_probe(int(probe_df.iloc[0]["index"]))

## 3. Load Policies Like The Server

In [ ]:
from lerobot.policies.factory import get_policy_class, make_pre_post_processors

policy_cls = get_policy_class("smolvla")

def assert_complete_model_snapshot(model_dir: Path, label: str):
    required = ["model.safetensors", "config.json", "train_config.json", "policy_preprocessor.json", "policy_postprocessor.json"]
    missing = [name for name in required if not (model_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"{label} model snapshot is incomplete: {missing}. Use the complete/postprocess commit.")

def load_policy_bundle(model_dir: Path, device: str, label: str):
    assert_complete_model_snapshot(model_dir, label)
    policy = policy_cls.from_pretrained(str(model_dir))
    policy.config.device = device
    policy.to(device)
    policy.eval()
    pre, post = make_pre_post_processors(
        policy.config,
        pretrained_path=str(model_dir),
        preprocessor_overrides={"device_processor": {"device": device}},
        postprocessor_overrides={"device_processor": {"device": device}},
    )
    print("loaded", label, model_dir)
    print("chunk_size", policy.config.chunk_size, "n_action_steps", policy.config.n_action_steps, "num_steps", getattr(policy.config, "num_steps", None))
    print("input_features", list(policy.config.input_features))
    return policy, pre, post

new_policy, new_pre, new_post = load_policy_bundle(new_model_dir, DEVICE, "new")
old_policy, old_pre, old_post = load_policy_bundle(old_model_dir, DEVICE, "old")

## 4. Inference Helpers

`processed` là action chunk đã qua postprocessor, cùng đơn vị joint/state thật. Đây là thứ cần plot để xem trajectory có tách mode không.

In [ ]:
def build_obs_for_policy(item: dict, policy, task: str) -> dict:
    obs = {}
    for key in policy.config.input_features:
        if key in item:
            obs[key] = item[key]
    obs["task"] = task
    return obs

def postprocess_chunk(raw_action_tensor: torch.Tensor, postprocessor) -> torch.Tensor:
    processed_actions = []
    _, chunk_size, _ = raw_action_tensor.shape
    for i in range(chunk_size):
        processed_action = postprocessor(raw_action_tensor[:, i, :])
        processed_actions.append(processed_action)
    return torch.stack(processed_actions, dim=1).squeeze(0).detach().cpu()

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def infer_once(policy, preprocessor, postprocessor, item: dict, task: str, seed: int):
    set_seed(seed)
    if hasattr(policy, "reset"):
        policy.reset()
    obs = build_obs_for_policy(item, policy, task)
    batch = preprocessor(obs)
    raw_norm = policy.predict_action_chunk(batch)
    if raw_norm.ndim != 3:
        raw_norm = raw_norm.unsqueeze(0)
    processed = postprocess_chunk(raw_norm, postprocessor)
    return raw_norm.detach().cpu(), processed

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Run Multi-Seed Probe

Nếu cùng một `episode/frame` mà các sample tách thành nhiều cụm trajectory rõ, đó là bằng chứng mạnh cho lỗi mode selection / stochastic instability.

In [ ]:
def run_many(policy, pre, post, item, label: str, episode_index: int, global_index: int):
    state = item["observation.state"].detach().cpu().numpy().astype(np.float64)
    rows = []
    chunks = []
    raw_chunks = []
    for i in range(N_SAMPLES):
        seed = SEED_BASE + i
        raw, proc_t = infer_once(policy, pre, post, item, TASK, seed)
        proc = proc_t.numpy().astype(np.float64)
        raw_np = raw.squeeze(0).numpy().astype(np.float64)
        chunks.append(proc)
        raw_chunks.append(raw_np)
        first = proc[0]
        rows.append({
            "model": label,
            "episode_index": episode_index,
            "global_index": global_index,
            "seed": seed,
            "first_action_dist_state": float(np.linalg.norm(first - state)),
            "first_action_dist_safe": float(np.linalg.norm(first - SAFE_POSE)),
            "chunk_mean_dist_state": float(np.mean(np.linalg.norm(proc - state, axis=1))),
            "chunk_mean_dist_safe": float(np.mean(np.linalg.norm(proc - SAFE_POSE, axis=1))),
            "chunk_endpoint_dist_state": float(np.linalg.norm(proc[-1] - state)),
            "chunk_endpoint_dist_safe": float(np.linalg.norm(proc[-1] - SAFE_POSE)),
            "chunk_mean_speed": float(np.mean(np.linalg.norm(np.diff(proc, axis=0), axis=1))),
            "first_gripper": float(first[-1]),
            "chunk_gripper_mean": float(proc[:, -1].mean()),
        })
    return pd.DataFrame(rows), np.stack(chunks), np.stack(raw_chunks), state

all_rows = []
chunk_store = {}
state_store = {}

for _, row in probe_df.iterrows():
    ep = int(row["episode_index"])
    idx = int(row["index"])
    item = ds[idx]
    for label, policy, pre, post in [
        ("new", new_policy, new_pre, new_post),
        ("old", old_policy, old_pre, old_post),
    ]:
        print("running", label, "episode", ep, "global", idx)
        df_part, chunks, raw_chunks, state = run_many(policy, pre, post, item, label, ep, idx)
        all_rows.append(df_part)
        chunk_store[(label, ep)] = chunks
        chunk_store[(label, ep, "raw_norm")] = raw_chunks
        state_store[ep] = state
        cleanup_cuda()

results = pd.concat(all_rows, ignore_index=True)
results.head()

In [ ]:
summary = results.groupby(["episode_index", "model"])[[
    "first_action_dist_state",
    "first_action_dist_safe",
    "chunk_mean_dist_state",
    "chunk_mean_dist_safe",
    "chunk_mean_speed",
    "first_gripper",
]].agg(["mean", "std", "min", "max"]).round(3)
summary

## 6. Tiny K-Means + PCA Helpers

Không phụ thuộc sklearn. Feature cluster mặc định là `CLUSTER_STEPS` timestep đầu, bỏ gripper để tập trung vào arm trajectory. Nếu muốn tính cả gripper, đổi `use_arm_only=False`.

In [ ]:
def chunk_features(chunks: np.ndarray, steps: int = CLUSTER_STEPS, use_arm_only: bool = True) -> np.ndarray:
    steps = min(steps, chunks.shape[1])
    x = chunks[:, :steps, :]
    if use_arm_only:
        x = x[:, :, :-1]
    return x.reshape(x.shape[0], -1)

def standardize(x: np.ndarray):
    mu = x.mean(axis=0, keepdims=True)
    sd = x.std(axis=0, keepdims=True)
    sd[sd < 1e-6] = 1.0
    return (x - mu) / sd, mu, sd

def pca2(x: np.ndarray):
    xz, _, _ = standardize(x)
    u, s, vt = np.linalg.svd(xz, full_matrices=False)
    return xz @ vt[:2].T

def tiny_kmeans(x: np.ndarray, k: int = N_CLUSTERS, n_iter: int = 80, seed: int = 0):
    xz, _, _ = standardize(x)
    rng = np.random.default_rng(seed)
    centers = xz[rng.choice(len(xz), size=k, replace=False)].copy()
    labels = np.zeros(len(xz), dtype=np.int64)
    for _ in range(n_iter):
        d = ((xz[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
        new_labels = d.argmin(axis=1)
        if np.array_equal(new_labels, labels):
            break
        labels = new_labels
        for j in range(k):
            if np.any(labels == j):
                centers[j] = xz[labels == j].mean(axis=0)
    return labels

def label_clusters_for(model: str, ep: int, k: int = N_CLUSTERS):
    chunks = chunk_store[(model, ep)]
    feats = chunk_features(chunks)
    return tiny_kmeans(feats, k=k, seed=42), pca2(feats)

## 7. Visualize Modes For One Episode

Đổi `EP_TO_PLOT` để soi từng start state. Nếu new model có nhiều cụm rõ hơn old trên cùng observation, đây là nghi phạm rất mạnh.

In [ ]:
EP_TO_PLOT = int(probe_df.iloc[0]["episode_index"])

def plot_pca_modes(ep: int):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model in zip(axes, ["new", "old"]):
        labels, xy = label_clusters_for(model, ep)
        sc = ax.scatter(xy[:, 0], xy[:, 1], c=labels, cmap="tab10", s=28, alpha=0.85)
        ax.set_title(f"{model} episode {ep}: PCA of first {CLUSTER_STEPS} action steps")
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_pca_modes(EP_TO_PLOT)

In [ ]:
def cluster_summary(model: str, ep: int):
    labels, _ = label_clusters_for(model, ep)
    sub = results[(results["model"] == model) & (results["episode_index"] == ep)].copy()
    sub["cluster"] = labels
    return sub.groupby("cluster")[[
        "seed",
        "first_action_dist_state",
        "first_action_dist_safe",
        "chunk_mean_dist_state",
        "chunk_mean_dist_safe",
        "chunk_mean_speed",
        "first_gripper",
    ]].agg({
        "seed": "count",
        "first_action_dist_state": "mean",
        "first_action_dist_safe": "mean",
        "chunk_mean_dist_state": "mean",
        "chunk_mean_dist_safe": "mean",
        "chunk_mean_speed": "mean",
        "first_gripper": "mean",
    }).rename(columns={"seed": "n"}).round(3)

display(cluster_summary("new", EP_TO_PLOT))
display(cluster_summary("old", EP_TO_PLOT))

In [ ]:
def plot_joint_trajectories_by_cluster(model: str, ep: int, max_lines_per_cluster: int = 25):
    chunks = chunk_store[(model, ep)]
    labels, _ = label_clusters_for(model, ep)
    state = state_store[ep]
    t = np.arange(chunks.shape[1])
    n_joints = chunks.shape[2]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()
    colors = plt.get_cmap("tab10")
    rng = np.random.default_rng(0)
    for j, ax in enumerate(axes[:n_joints]):
        for c in sorted(np.unique(labels)):
            idx = np.where(labels == c)[0]
            if len(idx) > max_lines_per_cluster:
                idx = rng.choice(idx, size=max_lines_per_cluster, replace=False)
            for ii in idx:
                ax.plot(t, chunks[ii, :, j], color=colors(c), alpha=0.12, linewidth=1)
            mean = chunks[labels == c, :, j].mean(axis=0)
            ax.plot(t, mean, color=colors(c), linewidth=2.5, label=f"cluster {c} n={(labels == c).sum()}")
        ax.axhline(SAFE_POSE[j], color="red", linestyle="--", linewidth=1, alpha=0.7, label="safe" if j == 0 else None)
        ax.axhline(state[j], color="black", linestyle=":", linewidth=1, alpha=0.7, label="current" if j == 0 else None)
        ax.set_title(JOINT_NAMES[j] if j < len(JOINT_NAMES) else f"joint {j}")
        ax.grid(alpha=0.25)
    axes[0].legend(loc="best")
    fig.suptitle(f"{model} episode {ep}: action chunks by cluster")
    plt.tight_layout()
    plt.show()

plot_joint_trajectories_by_cluster("new", EP_TO_PLOT)
plot_joint_trajectories_by_cluster("old", EP_TO_PLOT)

## 8. Scan All Probe Episodes

Bảng này giúp tìm episode nào có mode instability mạnh nhất. `pca_spread` càng lớn nghĩa là cùng một obs sinh ra chunk phân tán hơn trong feature space.

In [ ]:
scan_rows = []
for ep in sorted(probe_df["episode_index"].astype(int).unique()):
    for model in ["new", "old"]:
        chunks = chunk_store[(model, ep)]
        feats = chunk_features(chunks)
        xy = pca2(feats)
        labels = tiny_kmeans(feats, k=N_CLUSTERS, seed=42)
        counts = np.bincount(labels, minlength=N_CLUSTERS)
        probs = counts / counts.sum()
        entropy = float(-(probs[probs > 0] * np.log2(probs[probs > 0])).sum())
        scan_rows.append({
            "episode_index": ep,
            "model": model,
            "pca_spread": float(np.mean(np.linalg.norm(xy - xy.mean(axis=0), axis=1))),
            "cluster_entropy": entropy,
            "cluster_counts": counts.tolist(),
            "mean_first_dist_state": float(results[(results.model == model) & (results.episode_index == ep)]["first_action_dist_state"].mean()),
            "std_first_dist_state": float(results[(results.model == model) & (results.episode_index == ep)]["first_action_dist_state"].std()),
            "mean_first_dist_safe": float(results[(results.model == model) & (results.episode_index == ep)]["first_action_dist_safe"].mean()),
            "std_first_dist_safe": float(results[(results.model == model) & (results.episode_index == ep)]["first_action_dist_safe"].std()),
        })

scan = pd.DataFrame(scan_rows).round(3)
scan.sort_values(["episode_index", "model"])

## 9. Phase Sweep Across Many Frames

Phần này chạy nhiều frame đầu episode để tránh kết luận từ đúng một observation. Mỗi observation dùng ít seed hơn (`PHASE_SWEEP_N_SAMPLES`) để notebook vẫn chạy được trong một lần.

Mặc định sweep các frame `[0, 5, 10, 15, 20, 25, 30, 45, 60]`, tương đương khoảng 0-4s ở 15Hz.

In [ ]:
phase_rows = []
for ep in PHASE_SWEEP_EPISODES:
    for frame_idx in PHASE_SWEEP_FRAME_INDICES:
        rows = data[(data["episode_index"] == ep) & (data["frame_index"] == frame_idx)]
        if rows.empty:
            continue
        row = rows.iloc[0]
        phase_rows.append({
            "episode_index": int(row["episode_index"]),
            "frame_index": int(row["frame_index"]),
            "global_index": int(row["index"]),
            "timestamp": float(row["timestamp"]),
        })

phase_probe_df = pd.DataFrame(phase_rows)
print("phase observations", len(phase_probe_df))
phase_probe_df.head(20)

In [ ]:
def run_many_for_sweep(policy, pre, post, item, label: str, episode_index: int, frame_index: int, global_index: int):
    state = item["observation.state"].detach().cpu().numpy().astype(np.float64)
    rows = []
    chunks = []
    for i in range(PHASE_SWEEP_N_SAMPLES):
        seed = SEED_BASE + 100_000 + episode_index * 1_000 + frame_index * 10 + i
        _, proc_t = infer_once(policy, pre, post, item, TASK, seed)
        proc = proc_t.numpy().astype(np.float64)
        chunks.append(proc)
        first = proc[0]
        rows.append({
            "model": label,
            "episode_index": episode_index,
            "frame_index": frame_index,
            "global_index": global_index,
            "seed": seed,
            "state_dist_safe": float(np.linalg.norm(state - SAFE_POSE)),
            "first_action_dist_state": float(np.linalg.norm(first - state)),
            "first_action_dist_safe": float(np.linalg.norm(first - SAFE_POSE)),
            "chunk_mean_dist_state": float(np.mean(np.linalg.norm(proc - state, axis=1))),
            "chunk_mean_dist_safe": float(np.mean(np.linalg.norm(proc - SAFE_POSE, axis=1))),
            "chunk_endpoint_dist_state": float(np.linalg.norm(proc[-1] - state)),
            "chunk_endpoint_dist_safe": float(np.linalg.norm(proc[-1] - SAFE_POSE)),
            "chunk_mean_speed": float(np.mean(np.linalg.norm(np.diff(proc, axis=0), axis=1))),
            "first_gripper": float(first[-1]),
        })
    return pd.DataFrame(rows), np.stack(chunks), state

phase_sample_rows = []
phase_chunk_store = {}
phase_state_store = {}

for _, row in phase_probe_df.iterrows():
    ep = int(row["episode_index"])
    frame_idx = int(row["frame_index"])
    idx = int(row["global_index"])
    item = ds[idx]
    for label, policy, pre, post in [
        ("new", new_policy, new_pre, new_post),
        ("old", old_policy, old_pre, old_post),
    ]:
        print("phase", label, "ep", ep, "frame", frame_idx, "global", idx)
        df_part, chunks, state = run_many_for_sweep(policy, pre, post, item, label, ep, frame_idx, idx)
        phase_sample_rows.append(df_part)
        phase_chunk_store[(label, ep, frame_idx)] = chunks
        phase_state_store[(ep, frame_idx)] = state
        cleanup_cuda()

phase_samples = pd.concat(phase_sample_rows, ignore_index=True)
phase_samples.head()

In [ ]:
def chunk_instability_metrics(chunks: np.ndarray):
    feats = chunk_features(chunks)
    xy = pca2(feats)
    labels = tiny_kmeans(feats, k=N_CLUSTERS, seed=42)
    counts = np.bincount(labels, minlength=N_CLUSTERS)
    probs = counts / counts.sum()
    entropy = float(-(probs[probs > 0] * np.log2(probs[probs > 0])).sum())
    centroids = []
    for c in range(N_CLUSTERS):
        if np.any(labels == c):
            centroids.append(xy[labels == c].mean(axis=0))
    if len(centroids) >= 2:
        centroids = np.stack(centroids)
        centroid_gap = float(np.max(np.linalg.norm(centroids[:, None, :] - centroids[None, :, :], axis=2)))
    else:
        centroid_gap = 0.0
    return {
        "pca_spread": float(np.mean(np.linalg.norm(xy - xy.mean(axis=0), axis=1))),
        "cluster_entropy": entropy,
        "cluster_centroid_gap": centroid_gap,
        "cluster_counts": counts.tolist(),
    }

phase_summary_rows = []
for (model, ep, frame_idx), chunks in phase_chunk_store.items():
    sub = phase_samples[(phase_samples["model"] == model) & (phase_samples["episode_index"] == ep) & (phase_samples["frame_index"] == frame_idx)]
    state_dist_safe = float(sub["state_dist_safe"].iloc[0])
    metrics = chunk_instability_metrics(chunks)
    phase_summary_rows.append({
        "model": model,
        "episode_index": ep,
        "frame_index": frame_idx,
        "timestamp": float(sub["frame_index"].iloc[0]) / 15.0,
        "state_dist_safe": state_dist_safe,
        "mean_first_dist_state": float(sub["first_action_dist_state"].mean()),
        "std_first_dist_state": float(sub["first_action_dist_state"].std()),
        "mean_first_dist_safe": float(sub["first_action_dist_safe"].mean()),
        "std_first_dist_safe": float(sub["first_action_dist_safe"].std()),
        "mean_chunk_speed": float(sub["chunk_mean_speed"].mean()),
        "stay_current_rate": float((sub["first_action_dist_state"] <= STAY_CURRENT_DIST_THRESH).mean()),
        "safe_like_rate": float((sub["first_action_dist_safe"] <= SAFE_LIKE_DIST_THRESH).mean()),
        "safe_pull_rate": float((sub["first_action_dist_safe"] <= (state_dist_safe - SAFE_PULL_MARGIN)).mean()),
        "overshoot_rate": float((sub["first_action_dist_state"] >= OVERSHOOT_FIRST_DIST_THRESH).mean()),
        **metrics,
    })

phase_summary = pd.DataFrame(phase_summary_rows)
phase_summary.round(3).sort_values(["episode_index", "frame_index", "model"]).head(30)

## 10. New-vs-Old Suspicious Observations

Bảng này xếp các observation mà new khác old mạnh nhất. Đây là nơi tìm frame để mở deep plot.

In [ ]:
new_s = phase_summary[phase_summary["model"] == "new"].copy()
old_s = phase_summary[phase_summary["model"] == "old"].copy()
compare = new_s.merge(old_s, on=["episode_index", "frame_index"], suffixes=("_new", "_old"))
for col in ["pca_spread", "cluster_entropy", "cluster_centroid_gap", "stay_current_rate", "safe_like_rate", "safe_pull_rate", "overshoot_rate", "mean_first_dist_state", "mean_first_dist_safe"]:
    compare[f"delta_{col}"] = compare[f"{col}_new"] - compare[f"{col}_old"]

compare["suspicion_score"] = (
    compare["delta_pca_spread"].clip(lower=0)
    + compare["delta_cluster_entropy"].clip(lower=0)
    + compare["delta_cluster_centroid_gap"].clip(lower=0)
    + 3.0 * compare["safe_pull_rate_new"]
    + 2.0 * compare["stay_current_rate_new"]
    + 2.0 * compare["overshoot_rate_new"]
)

cols = [
    "episode_index", "frame_index", "suspicion_score",
    "pca_spread_new", "pca_spread_old", "delta_pca_spread",
    "cluster_entropy_new", "cluster_entropy_old", "delta_cluster_entropy",
    "stay_current_rate_new", "safe_like_rate_new", "safe_pull_rate_new", "overshoot_rate_new",
    "mean_first_dist_state_new", "mean_first_dist_state_old",
    "mean_first_dist_safe_new", "mean_first_dist_safe_old",
    "cluster_counts_new", "cluster_counts_old",
]
compare[cols].round(3).sort_values("suspicion_score", ascending=False).head(25)

## 11. Heatmaps

Heatmap giúp nhìn nhanh lỗi tập trung ở episode/frame nào. Nếu new chỉ xấu ở vài frame, ta deep plot đúng frame đó thay vì đoán.

In [ ]:
def plot_heatmap(df: pd.DataFrame, value: str, title: str):
    piv = df.pivot(index="episode_index", columns="frame_index", values=value).sort_index()
    fig, ax = plt.subplots(figsize=(11, 5))
    im = ax.imshow(piv.values, aspect="auto", cmap="viridis")
    ax.set_xticks(np.arange(len(piv.columns)))
    ax.set_xticklabels(piv.columns)
    ax.set_yticks(np.arange(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_xlabel("frame_index")
    ax.set_ylabel("episode_index")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

plot_heatmap(compare, "delta_pca_spread", "new - old PCA spread")
plot_heatmap(compare, "safe_pull_rate_new", "new safe-pull rate")
plot_heatmap(compare, "stay_current_rate_new", "new stay-current rate")
plot_heatmap(compare, "overshoot_rate_new", "new overshoot rate")

## 12. Deep Plot Any Suspicious Frame

Sau khi xem bảng/heatmap, đổi `DEEP_EP` và `DEEP_FRAME` sang observation nghi ngờ nhất để vẽ cluster trajectory chi tiết.

In [ ]:
top = compare.sort_values("suspicion_score", ascending=False).iloc[0]
DEEP_EP = int(top["episode_index"])
DEEP_FRAME = int(top["frame_index"])
print("default suspicious frame", DEEP_EP, DEEP_FRAME)

def plot_phase_deep(model: str, ep: int, frame_idx: int, max_lines_per_cluster: int = 25):
    chunks = phase_chunk_store[(model, ep, frame_idx)]
    labels = tiny_kmeans(chunk_features(chunks), k=N_CLUSTERS, seed=42)
    state = phase_state_store[(ep, frame_idx)]
    t = np.arange(chunks.shape[1])
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()
    colors = plt.get_cmap("tab10")
    rng = np.random.default_rng(0)
    for j, ax in enumerate(axes[:chunks.shape[2]]):
        for c in sorted(np.unique(labels)):
            idx = np.where(labels == c)[0]
            if len(idx) > max_lines_per_cluster:
                idx = rng.choice(idx, size=max_lines_per_cluster, replace=False)
            for ii in idx:
                ax.plot(t, chunks[ii, :, j], color=colors(c), alpha=0.12, linewidth=1)
            mean = chunks[labels == c, :, j].mean(axis=0)
            ax.plot(t, mean, color=colors(c), linewidth=2.5, label=f"cluster {c} n={(labels == c).sum()}")
        ax.axhline(SAFE_POSE[j], color="red", linestyle="--", linewidth=1, alpha=0.7, label="safe" if j == 0 else None)
        ax.axhline(state[j], color="black", linestyle=":", linewidth=1, alpha=0.7, label="current" if j == 0 else None)
        ax.set_title(JOINT_NAMES[j])
        ax.grid(alpha=0.25)
    axes[0].legend(loc="best")
    fig.suptitle(f"{model} episode {ep}, frame {frame_idx}: phase-sweep chunk clusters")
    plt.tight_layout()
    plt.show()

plot_phase_deep("new", DEEP_EP, DEEP_FRAME)
plot_phase_deep("old", DEEP_EP, DEEP_FRAME)

## 13. How To Read Results

- Nếu new có `pca_spread`/`cluster_entropy` cao hơn old rõ rệt trên cùng episode: nghiêng về lỗi **multi-mode / stochastic instability**.
- Nếu một cluster có `first_action_dist_safe` thấp và trajectory nằm gần safe/current line: đó là **safe-like/start-like mode**.
- Nếu một cluster đi xa khỏi current quá nhanh hoặc vượt qua vùng expected approach: đó là **overshoot mode**.
- Nếu phase sweep chỉ ra lỗi tập trung ở frame đầu episode: nghi start/initial phase hoặc reset distribution.
- Nếu phase sweep chỉ ra lỗi ở frame approach giữa episode: nghi data multi-style/timing hoặc perception/action mapping trong approach phase.
- Nếu old cũng multi-mode tương tự nhưng deploy old vẫn tốt hơn, cần kiểm tra aggregation/RTC/client khác giữa hai lần test.
- Nếu cả old/new offline đều ổn định nhưng robot thật vẫn fail ngẫu nhiên, lỗi có thể nằm ở runtime observation/preprocessing/camera latency/RTC/action queue.